In [1]:
!pip install -q "transformers<5" sentence-transformers faiss-cpu torch accelerate gradio pypdf python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 643.0 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 54.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.3 MB/s eta 0:00:00


In [2]:
import os, json, re
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/mariamessam47/legal-contracts-eg")
CONTRACTS_DIR = DATA_DIR
LAWS_DIR = DATA_DIR
LABELS_PATH = DATA_DIR / "risk_labels.json"

print(os.listdir(DATA_DIR) if DATA_DIR.exists() else "المسار غير موجود")

['labor_law_reference.txt', 'risk_labels.json', 'rent_normal_01.txt', 'rental_law_reference.txt', 'README.md', 'supply_normal_01.txt', 'employment_risky_01.txt', 'rent_risky_01.txt', 'employment_normal_01.txt', 'supply_risky_01.txt', 'civil_commercial_reference.txt']


In [3]:
CONTRACT_FILE_PREFIXES = ("rent_", "employment_", "supply_")

def load_contracts(contracts_dir):
    contracts = {}
    for file in sorted(Path(contracts_dir).glob("*.txt")):
        if file.name.startswith(CONTRACT_FILE_PREFIXES):
            with open(file, "r", encoding="utf-8") as f:
                contracts[file.stem] = f.read()
    return contracts

contracts = load_contracts(CONTRACTS_DIR)
print(f"تم تحميل {len(contracts)} عقد:", list(contracts.keys()))

تم تحميل 6 عقد: ['employment_normal_01', 'employment_risky_01', 'rent_normal_01', 'rent_risky_01', 'supply_normal_01', 'supply_risky_01']


In [4]:
def split_into_clauses(contract_text):
    pattern = r'(?m)^(البند\s+\S+)'
    parts = re.split(pattern, contract_text)

    clauses = []
    current_title = None
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if re.match(r'^البند\s+\S+', part):
            current_title = part
        elif current_title:
            clauses.append({"title": current_title, "text": part})
            current_title = None

    return clauses

In [5]:
from transformers import pipeline

zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

CONTRACT_LABELS = {
    "rental": "عقد إيجار",
    "employment": "عقد عمل",
    "supply": "عقد توريد بضائع"
}

CONTRACT_KEYWORDS = {
    "rental": ["مؤجر", "مستأجر", "الأجرة", "إيجار", "العين المؤجرة", "الإخلاء", "التأمين النقدي", "الوحدة", "المحل", "الشقة"],
    "employment": ["صاحب العمل", "العامل", "الموظف", "الأجر الشهري", "فترة الاختبار", "الإجازة", "ساعات العمل", "مكافأة نهاية الخدمة", "الفصل"],
    "supply": ["المورد", "التوريد", "الشحنة", "المشتري", "البضاعة", "المطابقة", "غرامة التأخير", "الفحص"]
}

def classify_contract_type(text):
    keyword_scores = {}
    for contract_key, keywords in CONTRACT_KEYWORDS.items():
        count = sum(text.count(kw) for kw in keywords)
        keyword_scores[contract_key] = count

    best_key = max(keyword_scores, key=keyword_scores.get)
    best_count = keyword_scores[best_key]
    total_count = sum(keyword_scores.values())

    if total_count > 0 and best_count / total_count >= 0.5:
        confidence = min(0.95, 0.6 + (best_count / total_count) * 0.4)
        return CONTRACT_LABELS[best_key], confidence, best_key

    result = zero_shot(text[:3000], list(CONTRACT_LABELS.values()),
                        hypothesis_template="هذا النص عبارة عن {}.")
    top_label = result["labels"][0]
    reverse_labels = {v: k for k, v in CONTRACT_LABELS.items()}
    return top_label, result["scores"][0], reverse_labels.get(top_label)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [6]:
from sentence_transformers import SentenceTransformer
import faiss

LAW_FILE_TO_TYPE = {
    "rental_law_reference": "rental",
    "labor_law_reference": "employment",
    "civil_commercial_reference": "supply"
}

def load_laws(laws_dir):
    law_chunks = []
    for file in sorted(Path(laws_dir).glob("*.txt")):
        if file.stem not in LAW_FILE_TO_TYPE:
            continue
        with open(file, "r", encoding="utf-8") as f:
            text = f.read()
        articles = re.split(r"(المادة المرجعية\s*\d+[^:]*:)", text)
        current_title = None
        for part in articles:
            part = part.strip()
            if not part:
                continue
            if part.startswith("المادة المرجعية"):
                current_title = part
            elif current_title:
                law_chunks.append({
                    "source": file.stem,
                    "contract_type": LAW_FILE_TO_TYPE[file.stem],
                    "title": current_title,
                    "text": part
                })
                current_title = None
    return law_chunks

law_chunks = load_laws(LAWS_DIR)
print(f"عدد المواد القانونية المرجعية: {len(law_chunks)}")

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
all_embeddings = embedder.encode([c["text"] for c in law_chunks], convert_to_numpy=True, show_progress_bar=True)

# فهرس عام (لو نوع العقد مش معروف) + فهارس منفصلة لكل نوع
law_index = faiss.IndexFlatL2(all_embeddings.shape[1])
law_index.add(all_embeddings)

law_indices_by_type = {}
law_chunks_by_type = {}
for type_key in ["rental", "employment", "supply"]:
    idxs = [i for i, c in enumerate(law_chunks) if c["contract_type"] == type_key]
    if not idxs:
        continue
    type_embeddings = all_embeddings[idxs]
    idx = faiss.IndexFlatL2(type_embeddings.shape[1])
    idx.add(type_embeddings)
    law_indices_by_type[type_key] = idx
    law_chunks_by_type[type_key] = [law_chunks[i] for i in idxs]

print("فهارس منفصلة جاهزة لـ:", list(law_indices_by_type.keys()))

عدد المواد القانونية المرجعية: 21


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

فهارس منفصلة جاهزة لـ: ['rental', 'employment', 'supply']


In [7]:
def retrieve_relevant_law(clause_text, contract_type_key=None, k=2):
    query_emb = embedder.encode([clause_text], convert_to_numpy=True)

    if contract_type_key and contract_type_key in law_indices_by_type:
        idx = law_indices_by_type[contract_type_key]
        chunks = law_chunks_by_type[contract_type_key]
    else:
        idx = law_index
        chunks = law_chunks

    k = min(k, len(chunks))
    _, indices = idx.search(query_emb, k)
    return [chunks[i] for i in indices[0]]

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
def build_risk_prompt(clause_title, clause_text, retrieved_laws):
    laws_context = "\n".join([f"- {l['title']} {l['text']}" for l in retrieved_laws])
    return f"""أنت مساعد قانوني. مهمتك تحليل بند من عقد ومقارنته بالقواعد القانونية التالية فقط. رد بالعربية فقط.

البند:
{clause_title}: {clause_text}

القواعد القانونية ذات الصلة:
{laws_context}

رد فقط بصيغة JSON بدون أي نص إضافي، بالشكل التالي بالضبط:
{{"is_risky": true or false, "risk_level": "low" or "medium" or "high", "reason": "شرح مختصر بالعربية فقط", "suggested_rewrite": "صياغة بديلة أعدل للبند أو فارغة لو البند سليم"}}
"""

def parse_json_output(raw_text):
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

RISK_WEIGHTS = {"low": 5, "medium": 15, "high": 30}

RED_FLAG_PATTERNS = [
    (r"تنازل.{0,40}(حق|حقه).{0,30}(التقاضي|الاعتراض|اللجوء|القضاء)", "تنازل عن حق التقاضي أو الاعتراض القانوني"),
    (r"دون.{0,15}(حكم قضائي|إذن قضائي|حاجة لحكم)", "تجاوز إجراءات القضاء الواجبة"),
    (r"بالقوة الجبرية|بالقوة الذاتية", "استخدام القوة الذاتية بدلاً من الإجراءات القانونية"),
    (r"دون.{0,10}(إبداء|ذكر).{0,10}(سبب|أسباب)", "إنهاء أو فسخ بدون سبب مشروع"),
    (r"لا يحق.{0,20}(للمستأجر|للعامل|للمشتري).{0,20}(المطالبة|الاعتراض|رد)", "إسقاط حق أساسي لأحد الطرفين"),
]

def check_red_flags(clause_text):
    for pattern, reason in RED_FLAG_PATTERNS:
        if re.search(pattern, clause_text):
            return True, reason
    return False, None

In [10]:
def analyze_full_contract(text, max_new_tokens=400, progress_callback=None):
    contract_type, type_score, contract_type_key = classify_contract_type(text)
    clauses = split_into_clauses(text)
    if not clauses:
        return {"error": "لم يتم التعرف على بنود بصيغة 'البند ...' في هذا الملف."}

    rows = []
    risky_count = 0
    raw_risk_score = 0

    for i, clause in enumerate(clauses):
        is_red_flag, red_flag_reason = check_red_flags(clause["text"])

        retrieved = retrieve_relevant_law(clause["text"], contract_type_key=contract_type_key, k=2)
        prompt = build_risk_prompt(clause["title"], clause["text"], retrieved)

        inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
        outputs = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0)
        raw = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        parsed = parse_json_output(raw)

        if is_red_flag:
            if not parsed:
                parsed = {}
            parsed["is_risky"] = True
            parsed["risk_level"] = parsed.get("risk_level") if parsed.get("risk_level") in ["medium", "high"] else "high"
            parsed["reason"] = f"[كشف تلقائي] {red_flag_reason}. " + parsed.get("reason", "")

        if parsed and parsed.get("is_risky"):
            risky_count += 1
            level = parsed.get("risk_level", "low")
            raw_risk_score += RISK_WEIGHTS.get(level, 5)
            icon = "🔴" if level == "high" else "🟡" if level == "medium" else "🟢"
            reason = parsed.get("reason", "")
            rewrite = parsed.get("suggested_rewrite", "")
        else:
            level, icon, reason, rewrite = "-", "✅", "لا توجد مخالفة" if parsed else "تعذر تحليل رد النموذج", ""

        rows.append([clause["title"], icon, level, reason, rewrite])

        if progress_callback:
            progress_callback((i + 1) / len(clauses), f"تم تحليل البند {i+1} من {len(clauses)}")

    max_possible_score = len(clauses) * RISK_WEIGHTS["high"]
    total_score = round((raw_risk_score / max_possible_score) * 100) if max_possible_score > 0 else 0
    total_score = min(total_score, 100)

    risk_label = "🔴 خطر مرتفع" if total_score >= 50 else "🟡 خطر متوسط" if total_score >= 20 else "🟢 خطر منخفض"

    return {
        "contract_type": contract_type,
        "type_score": type_score,
        "num_clauses": len(clauses),
        "risky_count": risky_count,
        "risk_score": total_score,
        "risk_label": risk_label,
        "rows": rows
    }

In [11]:
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

def evaluate_contract(contract_name, rows):
    gt = ground_truth[contract_name]
    gt_risky = {c["clause_number"] for c in gt["risky_clauses"]}
    predicted_risky = {i for i, row in enumerate(rows, start=1) if row[1] != "✅"}

    tp = len(gt_risky & predicted_risky)
    fp = len(predicted_risky - gt_risky)
    fn = len(gt_risky - predicted_risky)

    if len(gt_risky) == 0:
        correct = 1.0 if len(predicted_risky) == 0 else 0.0
        return {"precision": correct, "recall": 1.0, "f1": correct, "tp": tp, "fp": fp, "fn": fn}

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

In [12]:
all_metrics = {}
for name, text in contracts.items():
    result = analyze_full_contract(text)
    metrics = evaluate_contract(name, result["rows"])
    all_metrics[name] = metrics
    print(f"{name:25s} -> Precision={metrics['precision']:.2f}  Recall={metrics['recall']:.2f}  F1={metrics['f1']:.2f}")

avg_f1 = sum(m['f1'] for m in all_metrics.values()) / len(all_metrics)
print(f"\n📊 متوسط F1 على كل العقود: {avg_f1:.2f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


employment_normal_01      -> Precision=0.00  Recall=1.00  F1=0.00
employment_risky_01       -> Precision=1.00  Recall=0.71  F1=0.83
rent_normal_01            -> Precision=0.00  Recall=1.00  F1=0.00
rent_risky_01             -> Precision=0.83  Recall=0.71  F1=0.77
supply_normal_01          -> Precision=0.00  Recall=1.00  F1=0.00
supply_risky_01           -> Precision=1.00  Recall=1.00  F1=1.00

📊 متوسط F1 على كل العقود: 0.43


In [13]:

total_tp, total_fp, total_fn = 0, 0, 0

for name, text in contracts.items():
    result = analyze_full_contract(text)
    gt = ground_truth[name]
    gt_risky = {c["clause_number"] for c in gt["risky_clauses"]}
    predicted_risky = {i for i, row in enumerate(result["rows"], start=1) if row[1] != "✅"}

    total_tp += len(gt_risky & predicted_risky)
    total_fp += len(predicted_risky - gt_risky)
    total_fn += len(gt_risky - predicted_risky)

micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) else 0

print(f"إجمالي البنود الخطرة الحقيقية: {total_tp + total_fn}")
print(f"تم اكتشافها صح (TP): {total_tp}")
print(f"إيجابيات كاذبة (FP): {total_fp}")
print(f"سلبيات كاذبة فائتة (FN): {total_fn}")
print(f"\n📊 Micro-Precision: {micro_precision:.2f}")
print(f"📊 Micro-Recall: {micro_recall:.2f}")
print(f"📊 Micro-F1: {micro_f1:.2f}")

إجمالي البنود الخطرة الحقيقية: 19
تم اكتشافها صح (TP): 15
إيجابيات كاذبة (FP): 9
سلبيات كاذبة فائتة (FN): 4

📊 Micro-Precision: 0.62
📊 Micro-Recall: 0.79
📊 Micro-F1: 0.70


In [14]:
from pypdf import PdfReader
import docx

def extract_text_from_file(file_obj):
    file_path = file_obj.name if hasattr(file_obj, "name") else str(file_obj)

    if file_path.lower().endswith(".pdf"):
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        return text
    elif file_path.lower().endswith((".docx", ".doc")):
        doc = docx.Document(file_path)
        return "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

In [15]:
import gradio as gr

def gradio_analyze(file_obj, progress=gr.Progress()):
    if file_obj is None:
        return "من فضلك ارفع ملف عقد أولاً.", None, None

    try:
        progress(0.05, desc="جاري استخراج النص من الملف...")
        text = extract_text_from_file(file_obj)

        if not text.strip():
            return "لم يتم العثور على نص داخل الملف.", None, None

        def cb(frac, msg):
            progress(0.1 + 0.85 * frac, desc=msg)

        progress(0.1, desc="جاري تصنيف نوع العقد وتحليل البنود...")
        result = analyze_full_contract(text, progress_callback=cb)

        if "error" in result:
            return result["error"], None, None

        summary = f"""
### نتيجة تحليل العقد

**نوع العقد المكتشف:** {result['contract_type']} (ثقة {result['type_score']:.0%})
**عدد البنود:** {result['num_clauses']}
**عدد البنود الخطرة:** {result['risky_count']}
**درجة الخطورة الكلية:** {result['risk_score']}/100 — {result['risk_label']}
"""
        df = pd.DataFrame(result["rows"], columns=["البند", "الحالة", "درجة الخطورة", "السبب", "صياغة بديلة مقترحة"])

        report_path = "/tmp/contract_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        return summary, df, report_path

    except Exception as e:
        return f"حدث خطأ أثناء المعالجة: {str(e)}", None, None

In [16]:
custom_css = """
footer {visibility: hidden; display: none !important;}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="المساعد القانوني الذكي") as demo:
    gr.Markdown("""
    # ⚖️ المساعد القانوني الذكي لتحليل العقود
    ارفع عقد (**PDF**, **TXT**, أو **DOCX**) وسيتم تحليله تلقائيًا: تصنيف نوعه، كشف البنود المخالفة للقانون المصري،
    حساب درجة الخطورة، واقتراح صياغة بديلة للبنود الخطرة.
    """)

    with gr.Row():
        file_input = gr.File(label="ارفع ملف العقد (PDF, TXT, DOCX)", file_types=[".pdf", ".txt", ".docx", ".doc"])
        analyze_btn = gr.Button("🔍 حلل العقد", variant="primary")

    summary_output = gr.Markdown()
    table_output = gr.Dataframe(label="تحليل تفصيلي لكل بند", wrap=True)
    download_output = gr.File(label="تحميل التقرير كامل (JSON)")

    analyze_btn.click(fn=gradio_analyze, inputs=[file_input], outputs=[summary_output, table_output, download_output])

/tmp/ipykernel_58/693019348.py:5: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="المساعد القانوني الذكي") as demo:
/tmp/ipykernel_58/693019348.py:5: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="المساعد القانوني الذكي") as demo:


In [ ]:
demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://58f3ad2c6c456140f8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
